In [40]:
# Plotting RNA-seq TPM across conditions with multiple replicates

Import packages, files

In [41]:
%pip install plotly
%pip install kaleido
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import re

DEPRECATION: Loading egg at /Users/juliemcdonald/ENTER/lib/python3.12/site-packages/LoFreq_Star-2.1.5-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
Note: you may need to restart the kernel to use updated packages.
DEPRECATION: Loading egg at /Users/juliemcdonald/ENTER/lib/python3.12/site-packages/LoFreq_Star-2.1.5-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
Note: you may need to restart the kernel to use updated packages.


In [42]:
# --- INPUTS ---
# List of (file_path, sample_name) tuples
#final_gene_counts files you want to plot
files = [
    ("/Users/juliemcdonald/Documents/flamholz_lab/RNASeq_testing/two_part_snakefile/local_testing/switzer_lim_24h_R1/final_gene_counts.csv", "L20m_1"),
    ("/Users/juliemcdonald/Documents/flamholz_lab/RNASeq_testing/two_part_snakefile/local_testing/switzer_lim_24h_R2/final_gene_counts.csv", "L20m_2"),
    ("/Users/juliemcdonald/Documents/flamholz_lab/RNASeq_testing/two_part_snakefile/local_testing/switzer_WT_R1/final_gene_counts.csv", "R1"),
    ("/Users/juliemcdonald/Documents/flamholz_lab/RNASeq_testing/two_part_snakefile/local_testing/switzer_WT_R2/final_gene_counts.csv", "R2"),

]

#gene_to_organism map
gmap = '/Users/juliemcdonald/Documents/flamholz_lab/RNASeq_testing/two_part_snakefile/local_testing/gene_to_organism_map.tsv'



In [43]:
# --- Load and merge ---
META_COLS = ["gene_id", "Chr", "Product", "Name"]

merged = None
for path, name in files:
    df = pd.read_csv(path)
    df = df.rename(columns={"TPM": f"{name}_TPM"})

    if merged is None:
        # First file: keep all metadata columns
        merged = df[META_COLS + [f"{name}_TPM"]]
    else:
        # Subsequent files: only bring in gene_id + TPM column
        merged = pd.merge(merged, df[["gene_id", f"{name}_TPM"]], on="gene_id", how="inner")

# --- Save ---
tpm_cols = [f"{name}_TPM" for _, name in files]
merged = merged[META_COLS + tpm_cols]

# --- Filter: keep only genes that reach cutoff TPM in at least one sample ---
cutoff = 5

high_expr_mask = (merged[tpm_cols] > cutoff).any(axis=1)
merged = merged[high_expr_mask]

merged.to_csv("/Volumes/One_Touch/ncbi_data/palsson/palsson/merged_gene_TPM_replicates.csv", index=False)
print(f"Done! {len(merged)} genes written to merged_gene_TPM.csv")


Done! 3435 genes written to merged_gene_TPM.csv


# Calculate data for plot
Average, standard deviation of replicates \\
Log-corrected standard deviation \\
RANSAC regression \\
RSD of linear fit

In [44]:
%pip install scikit-learn
from sklearn.linear_model import RANSACRegressor
from sklearn import linear_model
import plotly.express as px
import plotly.graph_objects as go

# Get all TPM columns
tpm_cols = [col for col in merged.columns if col.endswith('TPM')]

# Keep only genes with counts > 0 in all TPM columns
mask = (merged[tpm_cols] > 0).all(axis=1)
df = merged[mask].copy()

mask = (merged[tpm_cols] < 98000).all(axis=1)
df = merged[mask].copy()

# Get all columns starting with L
l_cols = [col for col in df.columns if col.startswith('L')]

# Create average and standard deviation columns
df['L_avg'] = df[l_cols].mean(axis=1)
df['L_std'] = df[l_cols].std(axis=1)

# Get all columns starting with R
r_cols = [col for col in df.columns if col.startswith('R')]

# Create average and standard deviation columns
df['R_avg'] = df[r_cols].mean(axis=1)
df['R_std'] = df[r_cols].std(axis=1)

#Calculate relative error for symmetric error bars on log plot
#file:///Users/juliemcdonald/Downloads/EstimatingandPlottingLogarithmicErrorBars.pdf
df['R_rel_err'] = (df['R_std'] / df['R_avg']) / np.log(10)
df['L_rel_err'] = (df['L_std'] / df['L_avg']) / np.log(10)

#Choose what to plot
x_name = 'R_avg'
x_error = 'R_rel_err'
# x_error = 'R_std'

y_name = 'L_avg'
y_error = 'L_rel_err'
# y_error = 'L_std'

x = df[x_name]
y = df[y_name]
x_err = df[x_error]
y_err = df[y_error]

# Fit RANSAC regression on log10-transformed values
log_x = np.log10(x).values.reshape(-1, 1)
log_y = np.log10(y).values.reshape(-1, 1)

ransac = linear_model.RANSACRegressor(residual_threshold=1.0, random_state=100)
ransac.fit(log_x, log_y)

inlier_mask  = ransac.inlier_mask_
outlier_mask = np.logical_not(inlier_mask)

slope     = ransac.estimator_.coef_[0][0]
intercept = ransac.estimator_.intercept_[0]

# Generate fit line
x_fit = np.linspace(log_x.min(), log_x.max(), 200)
y_fit = slope * x_fit + intercept

#Calculate residuals & RSD threshold (on inliers only)
predicted        = slope * log_x.flatten() + intercept
residuals        = log_y.flatten() - predicted
rsd              = residuals[inlier_mask].std()
df["residual"]   = residuals

print(f"RANSAC slope={slope:.3f}, intercept={intercept:.3f}")
print(f"Inliers: {inlier_mask.sum()}  Outliers: {outlier_mask.sum()}")

outliers = df[outlier_mask].copy()
outliers['L/R'] = outliers['L_avg'] / outliers['R_avg']
outliers.to_csv('outliers.csv', index=False)
print(f"Done! {len(outliers)} genes written to outliers.csv")

#Mark unannotated outliers as any that are labeled as 'hypothetical protein' or 'putative' something

mask = (
    outliers['Product'].str.contains('hypothetical protein', case=False, na=False) |
    outliers['Product'].str.contains('putative', case=False, na=False)
)

unannotated = outliers[mask]
unannotated.to_csv('unannotated_outliers.csv', index=False)
print(f"Done! {len(unannotated)} rows written to unannotated_outliers.csv")


DEPRECATION: Loading egg at /Users/juliemcdonald/ENTER/lib/python3.12/site-packages/LoFreq_Star-2.1.5-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
Note: you may need to restart the kernel to use updated packages.
RANSAC slope=0.737, intercept=0.057
Inliers: 3251  Outliers: 180
Done! 180 genes written to outliers.csv
Done! 49 rows written to unannotated_outliers.csv


# Plotting with plotly

In [45]:
# Convert fit line back to linear scale for plotly
line_X_linear = 10**x_fit
line_y_linear = 10**y_fit

inliers  = df[inlier_mask]
outliers = df[outlier_mask]

fig = go.Figure()

# Inliers
fig.add_trace(go.Scatter(
    x=inliers[x_name], y=inliers[y_name],
    error_x=dict(type='data', array=inliers[x_error].values, visible=True),
    error_y=dict(type='data', array=inliers[y_error].values, visible=True),
    mode='markers',
    marker=dict(size=6, color='silver', opacity = 0.75,
                line=dict(width=.5, color='Black')),
    name='Inliers',
    customdata=inliers[['Name', 'gene_id', 'Product', 'Chr', x_name, y_name]].values,
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "gene_id: %{customdata[1]}<br>"
        "Product: %{customdata[2]}<br>"
        "Chr: %{customdata[3]}<br>"
        f"{x_name}: " + "%{customdata[4]:.2f}<br>"
        f"{y_name}: " + "%{customdata[5]:.2f}<extra></extra>"
    )
))

# Outliers
outliers_filtered = outliers[outliers['Product'] != 'hypothetical protein']

fig.add_trace(go.Scatter(
    x=outliers_filtered[x_name], y=outliers_filtered[y_name],
    error_x=dict(type='data', array=outliers_filtered[x_error].values, visible=True),
    error_y=dict(type='data', array=outliers_filtered[y_error].values, visible=True),
    mode='markers',
    marker=dict(size=6, color='dodgerblue', 
                line=dict(width=.5, color='Black')),
    name='Outliers',
    customdata=outliers_filtered[['Name', 'gene_id', 'Product', 'Chr', x_name, y_name]].values,
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "gene_id: %{customdata[1]}<br>"
        "Product: %{customdata[2]}<br>"
        "Chr: %{customdata[3]}<br>"
        f"{x_name}: " + "%{customdata[4]:.2f}<br>"
        f"{y_name}: " + "%{customdata[5]:.2f}<extra></extra>"
    )
))

#Unannotated outliers
fig.add_trace(go.Scatter(
    x=unannotated[x_name], y=unannotated[y_name],
    error_x=dict(type='data', array=unannotated[x_error].values, visible=True),
    error_y=dict(type='data', array=unannotated[y_error].values, visible=True),
    mode='markers',
    marker=dict(size=6, color='darkorange', symbol='diamond',
                line=dict(width=1, color='Black')),
    name='Unannotated outliers',
    customdata=unannotated[['Name', 'gene_id', 'Product', 'Chr', x_name, y_name]].values,
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "gene_id: %{customdata[1]}<br>"
        "Product: %{customdata[2]}<br>"
        "Chr: %{customdata[3]}<br>"
        f"{x_name}: " + "%{customdata[4]:.2f}<br>"
        f"{y_name}: " + "%{customdata[5]:.2f}<extra></extra>"
    )
))

# # RANSAC fit line
# fig.add_trace(go.Scatter(
#     x=line_X_linear, y=line_y_linear,
#     mode='lines',
#     line=dict(color='orange', width=2),
#     name=f'RANSAC fit' #<br> (slope={slope:.2f}, intercept={intercept:.2f})'
# ))

fig.update_layout(
    width=750, height=620,
    plot_bgcolor='white',
    title=f"R vs L gene TPM — {len(df):,} genes. Cutoff = {cutoff} TPM",

    xaxis=dict(type='log', showgrid=True, gridcolor='#eeeeee', title='TPM in N replete media',
            tickvals=[10**0, 10**1, 10**2, 10**3, 10**4, 10**5],
            exponentformat='power', showexponent='all',
            showline=True, linecolor='black', linewidth=1, mirror=True,
            minor=dict(ticks='')),
    yaxis=dict(type='log', showgrid=True, gridcolor="#eeeeee", title='TPM in N replete media',
            tickvals=[10**0, 10**1, 10**2, 10**3, 10**4, 10**5],
            exponentformat='power', showexponent='all',
            showline=True, linecolor='black', linewidth=1, mirror=True,
            minor=dict(ticks='')),

    font=dict(
    family="Arial, monospace",
    size=12,
    color="black"

    )
)


fig.show()
fig.write_html("scatter_wt_vs_lim.html")
print("Saved: scatter_wt_vs_lim.html")
#fig.write_image("scatter_wt_vs_lim.png", scale=3)


Saved: scatter_wt_vs_lim.html


# Volcano plot

In [46]:
# --- Pseudocount to avoid log(0) ---
PSEUDOCOUNT = 0.5

print(df)

# Mean TPM per condition
df["L4_mean"] = df[["L4_1_TPM", "L4_2_TPM"]].mean(axis=1)
df["R4_mean"] = df[["R4_1_TPM", "R4_2_TPM"]].mean(axis=1)

# Log2 fold change: R4 / L4  (positive = upregulated in Replete)
df["log2FC"] = np.log2(
    (df["R4_mean"] + PSEUDOCOUNT) / (df["L4_mean"] + PSEUDOCOUNT)
)

# Paired t-test across replicates (L4_1 vs R4_1, L4_2 vs R4_2)
pvalues = []
for _, row in df.iterrows():
    l4 = [row["L4_1_TPM"], row["L4_2_TPM"]]
    r4 = [row["R4_1_TPM"], row["R4_2_TPM"]]
    try:
        _, p = stats.ttest_rel(l4, r4)
    except Exception:
        p = 1.0
    pvalues.append(p)

df["pvalue"] = pvalues
df["neg_log10_p"] = -np.log10(df["pvalue"].clip(lower=1e-300))



# Summary stats
print(f"log2FC range:  {df['log2FC'].min():.2f}  to  {df['log2FC'].max():.2f}")
print(f"p-value range: {df['pvalue'].min():.2e}  to  {df['pvalue'].max():.2e}")

             gene_id           Chr Product Name    L20m_1_TPM  L20m_2_TPM  \
0     KHPLLONG_03314   cn_contig_1     NaN  NaN   8885.300000     2.56779   
1     KHPLLONG_03317   cn_contig_1     NaN  NaN   8300.500000     1.83413   
3     KHPLLONG_04216   cn_contig_1     NaN  NaN      1.611740    18.65260   
5     KHPLLONG_04426   cn_contig_1     NaN  NaN      1.294270    17.45160   
6     KHPLLONG_06326   cn_contig_1     NaN  NaN      0.561665    17.45160   
...              ...           ...     ...  ...           ...         ...   
4342  IEOEGPMD_04004  pa_contig_19     NaN  NaN      0.207052     9.64332   
4343  IEOEGPMD_05213  pa_contig_19     NaN  NaN   3397.090000     2.56779   
4344  IEOEGPMD_05539  pa_contig_19     NaN  NaN  35509.700000     6.96971   
4345  IEOEGPMD_05540  pa_contig_19     NaN  NaN  25059.500000     6.96971   
4346  IEOEGPMD_05808  pa_contig_19     NaN  NaN  17871.500000    10.27110   

           R1_TPM     R2_TPM         L_avg         L_std        R_avg  \
0 

KeyError: "None of [Index(['L4_1_TPM', 'L4_2_TPM'], dtype='object')] are in the [columns]"

In [47]:
# Adjust these thresholds as needed
FC_THRESHOLD  = 1.0   # |log2FC| cutoff
P_THRESHOLD   = 0.05  # p-value cutoff

def classify(row):
    sig   = row["pvalue"] < P_THRESHOLD
    up    = row["log2FC"] >  FC_THRESHOLD
    down  = row["log2FC"] < -FC_THRESHOLD
    if sig and up:   return "Up in Replete"
    if sig and down: return "Up in Limited"
    if sig:          return "Sig. (low FC)"
    return "Not Significant"

df["category"] = df.apply(classify, axis=1)

counts = df["category"].value_counts()
print("Gene counts per category:")
print(counts.to_string())

KeyError: 'pvalue'

In [ ]:
COLOR_MAP = {
    "Up in Replete":    "#E63946",   # red
    "Up in Limited":    "#457B9D",   # blue
    "Sig. (low FC)":    "#F4A261",   # orange
    "Not Significant":  "#AAAAAA",   # grey
}

ORDER = ["Up in Replete", "Up in Limited", "Sig. (low FC)", "Not Significant"]

fig = go.Figure()

for cat in ORDER:
    sub = df[df["category"] == cat]
    if sub.empty:
        continue

    # Build hover text
    hover = (
        "<b>" + sub["gene_id"].fillna("") + "</b><br>"
        + sub["Name"].fillna("").apply(lambda x: f"Name: {x}<br>" if x else "")
        + sub["Product"].fillna("unknown").apply(lambda x: f"Product: {x}<br>")
        + "Chr: "      + sub["Chr"].fillna("")                        + "<br>"
        + "log2FC: "   + sub["log2FC"].round(3).astype(str)           + "<br>"
        + "p-value: "  + sub["pvalue"].apply(lambda p: f"{p:.3e}")    + "<br>"
        + "L4 mean: "  + sub["L4_mean"].round(2).astype(str)          + " TPM<br>"
        + "R4 mean: "  + sub["R4_mean"].round(2).astype(str)          + " TPM"
    )

    fig.add_trace(go.Scatter(
        x=sub["log2FC"],
        y=sub["neg_log10_p"],
        mode="markers",
        name=f"{cat} (n={len(sub)})",
        marker=dict(
            color=COLOR_MAP[cat],
            size=5 if cat == "Not Significant" else 7,
            opacity=0.55 if cat == "Not Significant" else 0.85,
            line=dict(width=0.3, color="white"),
        ),
        text=hover,
        hovertemplate="%{text}<extra></extra>",
        customdata=sub[["gene_id", "log2FC", "pvalue"]].values,
    ))

# Threshold lines
p_line = -np.log10(P_THRESHOLD)
x_range = [df["log2FC"].min() - 0.5, df["log2FC"].max() + 0.5]

# Horizontal p-value line
fig.add_hline(
    y=p_line,
    line=dict(color="black", width=1, dash="dash"),
    annotation_text=f"p = {P_THRESHOLD}",
    annotation_position="right",
    annotation_font_size=11,
)

# Vertical FC lines
for xval in [-FC_THRESHOLD, FC_THRESHOLD]:
    fig.add_vline(
        x=xval,
        line=dict(color="black", width=1, dash="dot"),
    )

fig.update_layout(
    title=dict(
        text="Volcano Plot — Replete (R4) vs Limited (L4)",
        font=dict(size=18),
    ),
    xaxis=dict(
        title="log₂ Fold Change  (R4 / L4)",
        zeroline=True, zerolinewidth=1, zerolinecolor="lightgrey",
        gridcolor="#f0f0f0",
    ),
    yaxis=dict(
        title="−log₁₀(p-value)",
        gridcolor="#f0f0f0",
    ),
    legend=dict(
        title="Category",
        borderwidth=1,
        bordercolor="lightgrey",
        bgcolor="rgba(255,255,255,0.85)",
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    width=900,
    height=620,
    hoverlabel=dict(bgcolor="white", font_size=12),
)

fig.show()

# Add CDS Start/End to Outliers
Reads `gene_id` from the outliers CSV and looks up the CDS `start` and `end`
coordinates from the genome map TSV (GFF3-style format).

In [48]:
GFF_COLS = [
    "seqname", "source", "feature",
    "start", "end",
    "score", "strand", "frame",
    "attributes", "organism"
]

genome_map = pd.read_csv(
    gmap,
    sep="\t",
    header=None,
    names=GFF_COLS,
)

print(f"Loaded {len(genome_map):,} rows")
genome_map.head()

#get start, end, strand of each CDS from gmap
def extract_locus_tag(attr_str):
    """Parse locus_tag=VALUE from a GFF attributes string."""
    match = re.search(r'locus_tag=([^;]+)', str(attr_str))
    return match.group(1) if match else None

genome_map["locus_tag"] = genome_map["attributes"].apply(extract_locus_tag)

# Keep only CDS rows and drop any entries where locus_tag couldn't be parsed
cds = genome_map[genome_map["feature"] == "CDS"].copy()
cds = cds.dropna(subset=["locus_tag"])

print(f"{len(cds):,} CDS entries with a locus_tag")
cds[["locus_tag", "start", "end", "strand"]].head()

# If a gene appears on multiple contigs, keep all rows (merge will duplicate
# the outlier row — investigate those manually if needed).
cds_lookup = cds[["locus_tag", "start", "end", "strand"]].rename(
    columns={"start": "cds_start", "end": "cds_end"}
)

duplicates = cds_lookup[cds_lookup.duplicated("locus_tag", keep=False)]
if not duplicates.empty:
    print(f"⚠️  {duplicates['locus_tag'].nunique()} locus_tag(s) appear more than once in the genome map:")
    print(duplicates[["locus_tag", "cds_start", "cds_end"]].to_string(index=False))
else:
    print("✅  All locus_tags are unique in the genome map.")

#Load outliers
outliers = pd.read_csv("unannotated_outliers.csv")
print(f"Loaded {len(outliers)} outlier rows")
outliers.head()

#get start and end
annotated = outliers.merge(
    cds_lookup,
    left_on="gene_id",
    right_on="locus_tag",
    how="left",
).drop(columns=["locus_tag"])  # redundant with gene_id

print(f"Annotated dataframe: {len(annotated)} rows, {annotated.shape[1]} columns")
annotated.head()

#check for any unmatched IDs
missing = annotated[annotated["cds_start"].isna()]
if missing.empty:
    print("✅  All gene_ids were matched to the genome map.")
else:
    print(f"⚠️  {len(missing)} row(s) had no match in the genome map:")
    print(missing[["gene_id"]].to_string(index=False))


#Save
output_path = "unannotated_outliers_start_end.csv"
annotated.to_csv(output_path, index=False)
print(f"Saved → {output_path}")
print(f"Columns: {list(annotated.columns)}")

Loaded 16,489 rows
16,489 CDS entries with a locus_tag
✅  All locus_tags are unique in the genome map.
Loaded 49 outlier rows
Annotated dataframe: 49 rows, 19 columns
✅  All gene_ids were matched to the genome map.
Saved → unannotated_outliers_start_end.csv
Columns: ['gene_id', 'Chr', 'Product', 'Name', 'L20m_1_TPM', 'L20m_2_TPM', 'R1_TPM', 'R2_TPM', 'L_avg', 'L_std', 'R_avg', 'R_std', 'R_rel_err', 'L_rel_err', 'residual', 'L/R', 'cds_start', 'cds_end', 'strand']


# notes
Found an outlier BBFGCJEA_03655 that is 200x expressed in limited nitrogen. 
Gene is a hypothetical protein of 60 AA. Closest match is a hypothetical protein of 76 AA from another E. coli strain

>BBFGCJEA_03655
MSVRRILLTGITQNAPWRCLCVGRSVAVSVLTVSETKMRCLVSSRAAFTKIYVIFLLKVW

The closest matches using blast and DeepGO indicate that this protein is a queuine tRNA-ribosyltransferase (tgt)

I found another paper with a similar mechanism: "the first, found in the intracellular pathogen Chlamydia trachomatis, uses YhhQ and tRNA guanine transglycosylase (TGT) homologs that have changed substrate specificities to directly salvage q, mimicking the eukaryotic pathway."
https://www.pnas.org/doi/10.1073/pnas.1909604116

So it seems like it's a nucleotide salvage gene!

Let's see if it holds true for Switzer et al. 